# 01 · PostgreSQL en SQL pur

Interroger une base relationnelle **uniquement en SQL**, pas à pas : dans ce notebook (grâce à l'extension jupysql) ou dans **DataGrip** avec le script `sql/01_postgres_datagrip.sql`, qui contient exactement les mêmes requêtes et les mêmes explications.

> 🎯 **Cas d'usage** — La direction d'un réseau de **24 écoles** réparties en France, au Bénin et au Sénégal prépare son conseil d'administration. Elle vous demande un **premier état des lieux chiffré de l'année scolaire 2025-2026** : combien d'écoles, de classes et d'élèves inscrits dans chaque pays, et quel niveau scolaire (moyenne générale du 1er trimestre). Les données sont dans la base PostgreSQL « gestion scolaire » du réseau, où vous avez un accès en **lecture**. Votre livrable : un tableau de bord par pays, obtenu en une seule requête (partie 10).

**Ce que vous allez apprendre**
- vous repérer dans une base inconnue : tables, colonnes, volumes ;
- écrire des requêtes `SELECT` : choisir des colonnes, filtrer (`WHERE`), trier (`ORDER BY`), limiter (`LIMIT`) ;
- résumer des données avec des agrégats (`COUNT`, `AVG`…), `GROUP BY` et `HAVING` ;
- relier plusieurs tables avec `INNER JOIN` et `LEFT JOIN` ;
- manipuler des dates, créer des catégories avec `CASE WHEN`, interroger une vue ;
- assembler le tout dans une requête de tableau de bord, et vérifier les chiffres avant de les livrer.

**Durée indicative** : 35 min — parties 0 à 2 : 7 min · 3 et 4 : 8 min · 5 et 6 : 10 min · 7 à 9 : 6 min · 10 : 4 min · la partie 11 est un bonus.

**Prérequis** : le fichier `.env` rempli à la racine du dépôt (modèle : `.env.example`, valeurs données par l'enseignant). Aucune connaissance SQL préalable. Pour travailler dans DataGrip : la fiche `docs/connexion_datagrip.md`.

> ⚠️ **Règles du jeu** — La base est **partagée** par toute la promotion. Dans ce TP, on ne fait **que lire** : uniquement des requêtes `SELECT`, jamais `INSERT`, `UPDATE`, `DELETE`, `CREATE` ou `DROP`. Gardez des requêtes légères : un `LIMIT` ou un filtre, surtout sur la table `notes` (plus d'un million de lignes).

## 0. Se connecter à la base

Jupyter exécute normalement du Python. L'extension **jupysql** ajoute une commande « magique » : une cellule qui commence par `%%sql` est envoyée telle quelle à la base de données, et le résultat s'affiche sous forme de tableau. Après la cellule ci-dessous, tout le notebook est en SQL.

La cellule suivante est la **seule cellule Python** de connexion. Elle :
1. lit les identifiants dans le fichier `.env` (`load_dotenv`), pour ne jamais les écrire dans le code ni les publier sur GitHub ;
2. construit l'adresse de la base (`URL.create`) puis un **moteur** de connexion (`create_engine`, de la bibliothèque SQLAlchemy) ;
3. confie ce moteur à jupysql (`%sql engine`) et limite l'affichage à 20 lignes par résultat.

Elle n'affiche **rien** quand tout va bien : sa première ligne, `%%capture --no-stderr`, masque les messages d'information de jupysql mais laisse passer les erreurs. La requête 0.1 vérifie ensuite que la base répond. En cas d'erreur, lisez le message :
- `... not 'NoneType'` : le fichier `.env` est introuvable ou incomplet ;
- `password authentication failed` ou `Tenant or user not found` : mot de passe ou utilisateur erroné ;
- les autres cas sont décrits dans la section « Problèmes fréquents » de `docs/connexion_datagrip.md`.

> 💡 **Driver** (pilote) : la bibliothèque qui « parle » le protocole réseau de PostgreSQL. Ici, c'est `psycopg` (d'où `postgresql+psycopg`). Dans DataGrip, c'est un driver JDBC, téléchargé en un clic.

> 💡 **Pooler** : notre base est hébergée chez Supabase et passe par un *pooler*, un intermédiaire qui partage quelques connexions entre tous les étudiants (port 6543, mode « transaction »). Ce mode ne supporte pas les requêtes préparées à l'avance : `prepare_threshold=None` les désactive. Sans ce réglage, vous obtiendriez des erreurs `prepared statement ... already exists`.

In [1]:
%%capture --no-stderr
# masque les messages d'information de jupysql, mais laisse passer les erreurs
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

load_dotenv()  # lit le fichier .env : les identifiants restent hors du code

url = URL.create(
    "postgresql+psycopg",  # le SGBD (PostgreSQL) + le driver Python (psycopg)
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DATABASE"),
)
# prepare_threshold=None : pas de requêtes préparées (obligatoire avec le pooler en mode transaction)
engine = create_engine(url, connect_args={"prepare_threshold": None})

%load_ext sql
%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
%sql engine

### 0.1 · Première requête : la base répond-elle ?

`version()` est une fonction de PostgreSQL qui renvoie la version du serveur. Si vous voyez une ligne commençant par « PostgreSQL 17 », la connexion fonctionne.

In [2]:
%%sql
SELECT version();

1 rows affected.

version
"PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 13.2.0, 64-bit"


Au-dessus de chaque tableau, jupysql indique le nombre de lignes reçues : « 1 rows affected. ». Malgré le mot anglais *affected*, rien n'a été modifié dans la base (DataGrip, lui, écrit « 1 row retrieved »).

## 1. Se repérer dans la base

Avant toute question « métier », on fait le tour du propriétaire. PostgreSQL est un **SGBD relationnel** (système de gestion de base de données) : il range les données dans des **tables**, c'est-à-dire des tableaux avec une **ligne** par enregistrement (une école, un élève…) et une **colonne** par information (nom, date…). Les tables sont regroupées dans des **schémas** (des sortes de dossiers) : les nôtres sont dans le schéma `public`.

PostgreSQL décrit sa propre structure dans un catalogue nommé `information_schema`, que l'on interroge… avec du SQL.

### 1.1 · Les tables et les vues du schéma public

Une requête se lit presque comme une phrase : `SELECT` (les colonnes voulues) `FROM` (la table) `WHERE` (la condition) `ORDER BY` (l'ordre du résultat). Un texte entre apostrophes, comme `'public'`, est une valeur. Le point-virgule `;` termine la requête.

In [3]:
%%sql
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_type, table_name;

17 rows affected.

table_name,table_type
absences,BASE TABLE
annees_scolaires,BASE TABLE
bulletins,BASE TABLE
classes,BASE TABLE
ecoles,BASE TABLE
eleves,BASE TABLE
enseignants,BASE TABLE
enseignements,BASE TABLE
evaluations,BASE TABLE
inscriptions,BASE TABLE


15 tables (`BASE TABLE`) et 2 vues (`VIEW`, voir partie 9). Les noms parlent d'eux-mêmes : `pays`, `villes`, `ecoles`, `classes`, `eleves`, `inscriptions`, `bulletins`, `notes`…

### 1.2 · Les colonnes d'une table

Quelles informations a-t-on sur une école ? `information_schema.columns` liste les colonnes de chaque table avec leur **type** : `integer` (nombre entier), `character varying` (texte), `date`, `numeric` (nombre décimal), `boolean` (vrai/faux), `timestamp` (date + heure).

In [4]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'ecoles'
ORDER BY ordinal_position;

12 rows affected.

column_name,data_type
id,integer
nom,character varying
adresse,character varying
ville_id,integer
type_ecole,character varying
niveau_ecole,character varying
telephone,character varying
email,character varying
directeur,character varying
date_creation,date


### 1.3 · Combien de lignes ? Le réflexe COUNT(*)

Réflexe n°1 avant de récupérer des données : **compter**. `COUNT(*)` renvoie le nombre de lignes d'une table sans les transférer : c'est le serveur qui compte, seul le résultat (un nombre) voyage sur le réseau. `AS nb_notes` donne un nom, un **alias**, à la colonne du résultat.

In [5]:
%%sql
SELECT COUNT(*) AS nb_notes
FROM notes;

1 rows affected.

nb_notes
1181586


Plus de 1,18 million de notes : on n'écrira **jamais** `SELECT * FROM notes` sans filtre ni `LIMIT`, surtout à 30 sur la même base.

### 1.4 · Le volume de plusieurs tables d'un coup

Chaque parenthèse est une petite requête qui renvoie un seul nombre ; on les place côte à côte dans un même `SELECT` pour obtenir une ligne de synthèse.

In [6]:
%%sql
SELECT (SELECT COUNT(*) FROM pays)    AS nb_pays,
       (SELECT COUNT(*) FROM villes)  AS nb_villes,
       (SELECT COUNT(*) FROM ecoles)  AS nb_ecoles,
       (SELECT COUNT(*) FROM classes) AS nb_classes,
       (SELECT COUNT(*) FROM eleves)  AS nb_eleves;

1 rows affected.

nb_pays,nb_villes,nb_ecoles,nb_classes,nb_eleves
3,12,24,1306,5549


3 pays, 12 villes, 24 écoles. Les 1 306 classes couvrent **7 années scolaires** (de 2019-2020 à 2025-2026) : pour notre mission, il faudra filtrer sur 2025-2026.

**Comment les tables sont reliées.** Chaque table a une **clé primaire** `id` : un numéro unique par ligne. Une **clé étrangère** est une colonne qui contient l'`id` d'une ligne d'une autre table : par exemple, `villes.pays_id` contient l'`id` du pays de la ville. Ce sont ces clés qui permettront de relier les tables (partie 6). Les liens utiles pour la mission :

```
pays ──< villes ──< ecoles ──< classes ──< inscriptions >── eleves
        (pays_id)  (ville_id)  (ecole_id)  (classe_id, eleve_id, annee_scolaire_id)

bulletins        : une ligne par élève, par année et par trimestre
                   (moyenne_generale, rang, appreciation)
annees_scolaires : de 2019-2020 à 2025-2026
```

Lisez `──<` comme « un… pour plusieurs… » : un pays a plusieurs villes, une ville a plusieurs écoles, une école a plusieurs classes, etc. Le symbole `>──` est le même lien dans l'autre sens : un élève a plusieurs inscriptions (une par année). Entre parenthèses, sous chaque table : la clé étrangère qu'elle contient.

> 💡 Le schéma complet et commenté (clés, contraintes, index) est dans le fichier `docs/schema_postgres.sql` du dépôt : à lire, pas à exécuter. Dans DataGrip, clic droit sur le schéma `public` → **Diagrams → Show Diagram** dessine aussi toutes les tables et leurs liens.

## 2. SELECT : choisir les colonnes

`SELECT` est **la** commande de lecture. On choisit les colonnes à afficher, on peut les renommer ou en calculer de nouvelles, et on limite le nombre de lignes reçues.

### 2.1 · Toute une (petite) table

`*` signifie « toutes les colonnes ». C'est pratique pour découvrir une **petite** table comme `pays` (3 lignes), à éviter sur les grosses tables.

In [7]:
%%sql
SELECT *
FROM pays;

3 rows affected.

id,nom,code_iso,continent,created_at
1,France,FR,Europe,2026-03-06 17:43:01.547087
2,Bénin,BJ,Afrique,2026-03-06 17:43:01.547087
3,Sénégal,SN,Afrique,2026-03-06 17:43:01.547087


### 2.2 · Quelques colonnes, quelques lignes

Sur une table plus grande, on nomme les colonnes utiles et on ajoute `LIMIT n` pour ne recevoir que `n` lignes : c'est rapide et cela ménage le serveur partagé.

In [8]:
%%sql
SELECT nom, type_ecole, niveau_ecole, capacite_max
FROM ecoles
LIMIT 5;

5 rows affected.

nom,type_ecole,niveau_ecole,capacite_max
École Guilbert – Paris,public,lycee,959
École Denis – Paris,public,college,342
École Da Costa – Lyon,public,lycee,958
École Didier – Lyon,public,lycee,632
École Weber – Marseille,public,primaire,423


### 2.3 · Renommer et calculer une colonne

`AS` renomme une colonne du résultat. On peut aussi **calculer** une colonne : l'opérateur `||` colle deux textes bout à bout.

Remarquez qu'on ne sélectionne ni l'e-mail ni le téléphone des parents : on ne prend que ce dont la mission a besoin. C'est le principe de **minimisation** du RGPD vu en cours (les noms de cette base sont fictifs, mais on garde le bon réflexe).

In [9]:
%%sql
SELECT prenom || ' ' || nom AS eleve,
       date_naissance       AS ne_le,
       nationalite
FROM eleves
LIMIT 5;

5 rows affected.

eleve,ne_le,nationalite
Léon Noël,2004-10-12,Sénégalaise
Frédérique Lacroix,2004-01-11,Française
Josette Dumas,2005-11-09,Sénégalaise
Vincent Delaunay,2005-03-30,Béninoise
Nathalie Baudry,2003-06-06,Sénégalaise


## 3. WHERE : filtrer les lignes

`WHERE` ne garde que les lignes qui respectent une condition. Les textes et les dates s'écrivent entre apostrophes `'...'`, les nombres sans. Les opérateurs à connaître :
- `=` égal, `<>` différent, `<`, `>`, `<=`, `>=` ;
- `IN (...)` : la valeur fait partie d'une liste ;
- `BETWEEN a AND b` : entre deux bornes, incluses ;
- `LIKE` / `ILIKE` : le texte ressemble à un motif ;
- `IS NULL` : la valeur est absente ;
- `AND`, `OR` : combiner plusieurs conditions.

### 3.1 · L'année scolaire en cours

La colonne `est_active` est un **booléen** (`true` = vrai, `false` = faux). Quelle année scolaire est en cours ?

In [10]:
%%sql
SELECT id, libelle, date_debut, date_fin
FROM annees_scolaires
WHERE est_active = true;

1 rows affected.

id,libelle,date_debut,date_fin
7,2025-2026,2025-09-01,2026-07-31


> 💡 Retenez : l'année **2025-2026** a l'identifiant **7**. Les tables `classes`, `inscriptions` et `bulletins` ont une colonne `annee_scolaire_id` : pour la mission, on filtrera avec `annee_scolaire_id = 7`.

### 3.2 · Égal à un texte

Les écoles privées du réseau. Attention, la valeur stockée est exactement `'prive'`, sans accent : en SQL, `'privé'` ne donnerait aucune ligne. La partie 4 montre comment lister les valeurs possibles d'une colonne.

In [11]:
%%sql
SELECT nom, type_ecole, niveau_ecole
FROM ecoles
WHERE type_ecole = 'prive';

5 rows affected.

nom,type_ecole,niveau_ecole
École Collet – Porto-Novo,prive,college
École Muller – Parakou,prive,primaire
École Morel – Abomey-Calavi,prive,lycee
École Berger – Thiès,prive,primaire
École Poirier – Saint-Louis,prive,lycee


### 3.3 · Différent de, et deux conditions

Les lycées qui ne sont pas publics : avec `AND`, les deux conditions doivent être vraies en même temps.

In [12]:
%%sql
SELECT nom, type_ecole, capacite_max
FROM ecoles
WHERE niveau_ecole = 'lycee'
  AND type_ecole <> 'public';

5 rows affected.

nom,type_ecole,capacite_max
École Neveu – Bordeaux,communautaire,425
École Morel – Abomey-Calavi,prive,833
École Poirier – Saint-Louis,prive,579
École Laroche – Ziguinchor,communautaire,273
École Samson – Ziguinchor,communautaire,746


### 3.4 · Une liste de valeurs (IN)

Trois grandes villes du réseau, une par pays. `IN` évite d'écrire `nom = 'Paris' OR nom = 'Cotonou' OR nom = 'Dakar'`.

In [13]:
%%sql
SELECT id, nom, pays_id
FROM villes
WHERE nom IN ('Paris', 'Cotonou', 'Dakar');

3 rows affected.

id,nom,pays_id
1,Paris,1
5,Cotonou,2
9,Dakar,3


### 3.5 · Un intervalle (BETWEEN)

Les écoles qui ont entre 400 et 600 places (bornes incluses).

In [14]:
%%sql
SELECT nom, capacite_max
FROM ecoles
WHERE capacite_max BETWEEN 400 AND 600;

7 rows affected.

nom,capacite_max
École Weber – Marseille,423
École Neveu – Bordeaux,425
École Collet – Porto-Novo,420
École Descamps – Porto-Novo,589
École Ribeiro – Dakar,433
École Poirier – Saint-Louis,579
École Rémy – Saint-Louis,473


### 3.6 · Un motif de texte (LIKE, ILIKE)

`LIKE` compare un texte à un **motif** où `%` remplace n'importe quelle suite de caractères (et `_` un seul caractère). `LIKE` distingue majuscules et minuscules ; `ILIKE` (propre à PostgreSQL) ne les distingue pas. Le nom de chaque école se termine par sa ville : cherchons les écoles de Dakar.

In [15]:
%%sql
SELECT nom
FROM ecoles
WHERE nom ILIKE '%dakar%';

2 rows affected.

nom
École Morel – Dakar
École Ribeiro – Dakar


### 3.7 · Les valeurs manquantes (IS NULL)

`NULL` signifie « valeur absente ou inconnue ». Ce n'est pas une valeur comme les autres : `motif = NULL` ne trouve **rien**, il faut écrire `motif IS NULL` (ou `IS NOT NULL`). Combien d'absences n'ont aucun motif renseigné ?

In [16]:
%%sql
SELECT COUNT(*) AS absences_sans_motif
FROM absences
WHERE motif IS NULL;

1 rows affected.

absences_sans_motif
3966


### 3.8 · Combiner AND et OR (avec des parenthèses)

Les absences « à vérifier » : sans motif **ou** pour voyage, **et** non justifiées. Les parenthèses indiquent ce qui va ensemble, comme en calcul.

In [17]:
%%sql
SELECT COUNT(*) AS absences_a_verifier
FROM absences
WHERE (motif IS NULL OR motif = 'Voyage')
  AND justifiee = false;

1 rows affected.

absences_a_verifier
3191


> ⚠️ Sans les parenthèses, `AND` est évalué avant `OR` et la question devient « sans motif (justifiées ou non), ou bien pour voyage et non justifiées » : on obtient 5 573 lignes au lieu de 3 191. Essayez !

### ✍️ Exercice 1 — Les écoles primaires publiques

La direction veut la liste des écoles **primaires** et **publiques**, avec leur nombre de places (`capacite_max`). Colonnes utiles : `niveau_ecole` et `type_ecole`.

In [18]:
%%sql
-- ✅ Solution
SELECT nom, capacite_max
FROM ecoles
WHERE niveau_ecole = 'primaire'
  AND type_ecole = 'public';

3 rows affected.

nom,capacite_max
École Weber – Marseille,423
École Gilles – Cotonou,632
École Ribeiro – Dakar,433


## 4. ORDER BY et DISTINCT : trier et dédoublonner

Sans `ORDER BY`, l'ordre des lignes n'est **pas garanti** : il peut changer d'une exécution à l'autre. `ORDER BY colonne` trie du plus petit au plus grand (`ASC`, par défaut) ; `DESC` inverse l'ordre. Combiné avec `LIMIT`, on obtient un « top n ».

### 4.1 · Les 5 plus grandes écoles

On trie par capacité décroissante et on garde les 5 premières lignes.

In [19]:
%%sql
SELECT nom, capacite_max
FROM ecoles
ORDER BY capacite_max DESC
LIMIT 5;

5 rows affected.

nom,capacite_max
École Parent – Thiès,1087
École Perret – Cotonou,1028
École Guilbert – Paris,959
École Da Costa – Lyon,958
École Morel – Dakar,921


### 4.2 · Trier sur plusieurs colonnes

D'abord par type d'école, puis par nom à l'intérieur de chaque type.

In [20]:
%%sql
SELECT type_ecole, nom
FROM ecoles
ORDER BY type_ecole, nom
LIMIT 10;

10 rows affected.

type_ecole,nom
communautaire,École Guillou – Marseille
communautaire,École Laroche – Ziguinchor
communautaire,École Morel – Dakar
communautaire,École Neveu – Bordeaux
communautaire,École Perret – Cotonou
communautaire,École Samson – Ziguinchor
prive,École Berger – Thiès
prive,École Collet – Porto-Novo
prive,École Morel – Abomey-Calavi
prive,École Muller – Parakou


### 4.3 · Les valeurs distinctes d'une colonne

`DISTINCT` supprime les doublons du résultat : pratique pour connaître les valeurs possibles d'une colonne avant d'écrire un filtre.

In [21]:
%%sql
SELECT DISTINCT type_ecole
FROM ecoles
ORDER BY type_ecole;

3 rows affected.

type_ecole
communautaire
prive
public


### 4.4 · DISTINCT et les NULL

Les motifs d'absence possibles : le `NULL` apparaît comme une valeur à part entière, en dernier (PostgreSQL range les `NULL` après les autres valeurs). Il s'affiche `None` dans le notebook et `<null>` dans DataGrip.

In [22]:
%%sql
SELECT DISTINCT motif
FROM absences
ORDER BY motif;

4 rows affected.

motif
Événement familial
Maladie
Voyage
None


### ✍️ Exercice 2 — Les combinaisons type / niveau

Quelles combinaisons (`type_ecole`, `niveau_ecole`) existent dans le réseau ? Affichez chaque combinaison une seule fois, triée par type puis par niveau. Indice : `DISTINCT` s'applique à l'ensemble des colonnes du `SELECT`.

In [23]:
%%sql
-- ✅ Solution
SELECT DISTINCT type_ecole, niveau_ecole
FROM ecoles
ORDER BY type_ecole, niveau_ecole;

9 rows affected.

type_ecole,niveau_ecole
communautaire,college
communautaire,lycee
communautaire,primaire
prive,college
prive,lycee
prive,primaire
public,college
public,lycee
public,primaire


Les 9 combinaisons existent : chaque type d'école (public, privé, communautaire) est présent aux trois niveaux.

> 💡 Le schéma (`docs/schema_postgres.sql`) autorise aussi le niveau `'mixte'`, mais aucune école ne l'utilise : le schéma dit ce qui est **possible**, seules les données disent ce qui **existe**.

## 5. Agrégats : COUNT, AVG, MIN, MAX, GROUP BY, HAVING

Une **fonction d'agrégat** résume plusieurs lignes en une seule valeur : `COUNT` (nombre), `SUM` (somme), `AVG` (moyenne), `MIN`, `MAX`. `ROUND(x, 2)` arrondit à 2 décimales. C'est le cœur d'un état des lieux chiffré… et c'est le serveur qui calcule : seul le résultat voyage sur le réseau.

### 5.1 · Résumer toute une table

Sans `GROUP BY`, l'agrégat porte sur toute la table : on obtient une seule ligne.

In [24]:
%%sql
SELECT COUNT(*)                    AS nb_ecoles,
       MIN(capacite_max)           AS plus_petite,
       MAX(capacite_max)           AS plus_grande,
       ROUND(AVG(capacite_max), 1) AS capacite_moyenne
FROM ecoles;

1 rows affected.

nb_ecoles,plus_petite,plus_grande,capacite_moyenne
24,227,1087,636.5


### 5.2 · Un résumé par groupe (GROUP BY)

`GROUP BY` forme des groupes de lignes qui ont la même valeur, puis calcule l'agrégat **dans chaque groupe**. Règle d'or : toute colonne du `SELECT` qui n'est pas dans un agrégat doit figurer dans le `GROUP BY`.

In [25]:
%%sql
SELECT type_ecole, COUNT(*) AS nb_ecoles
FROM ecoles
GROUP BY type_ecole
ORDER BY nb_ecoles DESC;

3 rows affected.

type_ecole,nb_ecoles
public,13
communautaire,6
prive,5


### 5.3 · La moyenne générale par trimestre en 2025-2026

La table `bulletins` contient la `moyenne_generale` de chaque élève à chaque trimestre. On garde l'année 2025-2026 (`annee_scolaire_id = 7`, vu en partie 3), puis on regroupe par trimestre.

In [26]:
%%sql
SELECT trimestre,
       COUNT(*)                        AS nb_bulletins,
       ROUND(AVG(moyenne_generale), 2) AS moyenne
FROM bulletins
WHERE annee_scolaire_id = 7
GROUP BY trimestre
ORDER BY trimestre;

3 rows affected.

trimestre,nb_bulletins,moyenne
1,2209,10.92
2,2209,10.97
3,2209,11.01


2 209 bulletins par trimestre, soit un par élève inscrit cette année. Moyenne du 1er trimestre : 10,92/20. Gardez ces deux chiffres en tête : nous les recouperons avec le tableau de bord final.

### 5.4 · Filtrer les groupes (HAVING)

`WHERE` filtre les **lignes** avant le regroupement ; `HAVING` filtre les **groupes** après le calcul. Quelles classes de 2025-2026 comptent plus de 40 élèves inscrits ?

In [27]:
%%sql
SELECT classe_id, COUNT(*) AS nb_inscrits
FROM inscriptions
WHERE annee_scolaire_id = 7
GROUP BY classe_id
HAVING COUNT(*) > 40
ORDER BY nb_inscrits DESC, classe_id;

9 rows affected.

classe_id,nb_inscrits
597,63
465,61
866,61
1225,51
598,45
1222,43
1167,42
780,41
1306,41


La classe n°597 compte 63 élèves : c'est beaucoup ! Nous la retrouverons dans les parties 9 et 11.

### ✍️ Exercice 3 — Les spécialités les plus représentées

Comptez les enseignants (table `enseignants`) par `specialite` et ne gardez que les spécialités qui ont **au moins 25** enseignants, de la plus représentée à la moins représentée.

In [28]:
%%sql
-- ✅ Solution
SELECT specialite, COUNT(*) AS nb_enseignants
FROM enseignants
GROUP BY specialite
HAVING COUNT(*) >= 25
ORDER BY nb_enseignants DESC, specialite;

5 rows affected.

specialite,nb_enseignants
Anglais,30
Français,30
Arts Plastiques,26
Philosophie,25
Sciences de la Vie,25


## 6. Jointures : relier les tables

Le nom de la ville d'une école n'est pas dans la table `ecoles` : on n'y trouve que `ville_id`. Une **jointure** (`JOIN`) relie deux tables en associant les lignes dont les clés correspondent, ici `ecoles.ville_id = villes.id`.

On donne un **alias** court à chaque table (`e` pour `ecoles`, `v` pour `villes`) : on peut alors écrire `e.nom` (le nom de l'école) ou `v.nom` (le nom de la ville) sans ambiguïté.

### 6.1 · INNER JOIN : écoles et villes

`INNER JOIN` ne garde que les lignes qui ont une correspondance **des deux côtés**. La condition de jointure s'écrit après `ON`.

In [29]:
%%sql
SELECT e.nom AS ecole, v.nom AS ville
FROM ecoles AS e
INNER JOIN villes AS v ON v.id = e.ville_id
LIMIT 5;

5 rows affected.

ecole,ville
École Guilbert – Paris,Paris
École Denis – Paris,Paris
École Da Costa – Lyon,Lyon
École Didier – Lyon,Lyon
École Weber – Marseille,Marseille


### 6.2 · Deux jointures : écoles, villes, pays

On enchaîne les jointures comme les maillons d'une chaîne : école → ville → pays. Puis on compte les écoles de chaque pays : première brique du tableau de bord.

In [30]:
%%sql
SELECT p.nom AS pays, COUNT(*) AS nb_ecoles
FROM ecoles AS e
INNER JOIN villes AS v ON v.id = e.ville_id
INNER JOIN pays   AS p ON p.id = v.pays_id
GROUP BY p.nom
ORDER BY p.nom;

3 rows affected.

pays,nb_ecoles
Bénin,8
France,8
Sénégal,8


### 6.3 · LEFT JOIN : y a-t-il des villes sans école ?

`LEFT JOIN` garde **toutes** les lignes de la table de gauche (celle du `FROM`), même sans correspondance ; les colonnes de la table de droite valent alors `NULL`. D'où une technique classique pour trouver les « orphelins » : `LEFT JOIN`, puis `WHERE ... IS NULL`.

In [31]:
%%sql
SELECT v.nom AS ville_sans_ecole
FROM villes AS v
LEFT JOIN ecoles AS e ON e.ville_id = v.id
WHERE e.id IS NULL;

ville_sans_ecole


Résultat vide : chacune des 12 villes a au moins une école. Un résultat vide est aussi une information ! Cherchons d'autres absences de correspondance, du côté des classes.

### 6.4 · Les classes d'une école et leurs effectifs

Prenons l'École Weber – Marseille (`ecole_id = 5`) : ses classes de 2025-2026 et le nombre d'élèves inscrits dans chacune. `COUNT(i.id)` ne compte que les valeurs non `NULL` : une classe sans aucune inscription affiche donc 0. Avec un `INNER JOIN`, ces classes auraient tout simplement **disparu** du résultat, sans prévenir.

In [32]:
%%sql
SELECT c.nom AS classe, COUNT(i.id) AS nb_inscrits
FROM classes AS c
LEFT JOIN inscriptions AS i ON i.classe_id = c.id
WHERE c.ecole_id = 5
  AND c.annee_scolaire_id = 7
GROUP BY c.nom
ORDER BY c.nom;

10 rows affected.

classe,nb_inscrits
CE1 A,0
CE1 B,0
CE2 A,0
CE2 B,0
CM1 A,0
CM1 B,1
CM1 C,0
CM2 A,0
CM2 B,1
CP A,0


### 6.5 · Combien de classes vides, et dans quelles écoles ?

On combine les deux jointures : `INNER JOIN` vers `ecoles` (chaque classe a une école) et `LEFT JOIN` vers `inscriptions` avec `i.id IS NULL` pour ne garder que les classes **sans aucun inscrit**.

In [33]:
%%sql
SELECT e.nom AS ecole, e.niveau_ecole, COUNT(*) AS classes_sans_eleve
FROM classes AS c
INNER JOIN ecoles AS e ON e.id = c.ecole_id
LEFT JOIN inscriptions AS i ON i.classe_id = c.id
WHERE c.annee_scolaire_id = 7
  AND i.id IS NULL
GROUP BY e.nom, e.niveau_ecole
ORDER BY classes_sans_eleve DESC, e.nom;

6 rows affected.

ecole,niveau_ecole,classes_sans_eleve
École Berger – Thiès,primaire,11
École Muller – Parakou,primaire,11
École Gilles – Cotonou,primaire,9
École Ribeiro – Dakar,primaire,9
École Guillou – Marseille,primaire,8
École Weber – Marseille,primaire,8


> 💡 **Constat à remonter à la direction** : 56 classes ouvertes en 2025-2026 n'ont aucun élève inscrit (11 + 11 + 9 + 9 + 8 + 8), toutes dans les 6 écoles **primaires** (qui n'accueillent que 23 élèves à elles six). Classes fermées mais restées dans la base ? Inscriptions non saisies ? Le SQL ne tranche pas, mais c'est exactement le genre d'anomalie qu'un état des lieux doit signaler.

### ✍️ Exercice 4 — Les enseignants par pays

Combien d'enseignants travaillent dans chaque pays ? Reliez `enseignants` → `ecoles` (par `ecole_id`) → `villes` → `pays`, puis comptez par pays, du plus grand nombre au plus petit.

In [34]:
%%sql
-- ✅ Solution
SELECT p.nom AS pays, COUNT(*) AS nb_enseignants
FROM enseignants AS en
INNER JOIN ecoles AS e ON e.id = en.ecole_id
INNER JOIN villes AS v ON v.id = e.ville_id
INNER JOIN pays   AS p ON p.id = v.pays_id
GROUP BY p.nom
ORDER BY nb_enseignants DESC;

3 rows affected.

pays,nb_enseignants
Sénégal,102
France,97
Bénin,78


102 + 97 + 78 = 277 : on retrouve bien tous les enseignants de la table.

## 7. Travailler avec des dates

PostgreSQL sait calculer sur les dates. Les outils de base :
- `CURRENT_DATE` : la date du jour (le résultat dépend donc du jour où vous exécutez la requête) ;
- `EXTRACT(YEAR FROM une_date)` : extrait l'année (ou `MONTH` pour le mois, `DAY` pour le jour) ;
- `AGE(date_recente, date_ancienne)` : la durée écoulée entre deux dates, en années, mois et jours ;
- `date_trunc('month', une_date)` : ramène une date au 1er jour de son mois, idéal pour compter « par mois » ;
- `DATE '2025-09-01'` : une date écrite en dur, au format année-mois-jour.

### 7.1 · L'ancienneté des écoles

Pour chaque école : son année de création et son âge en années. `EXTRACT(YEAR FROM AGE(...))` ne garde que le nombre d'années entières de la durée.

In [35]:
%%sql
SELECT nom,
       date_creation,
       EXTRACT(YEAR FROM date_creation)                    AS annee_creation,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, date_creation)) AS anciennete_ans
FROM ecoles
ORDER BY date_creation
LIMIT 5;

5 rows affected.

nom,date_creation,annee_creation,anciennete_ans
École Guilbert – Paris,1981-02-13,1981,45
École Descamps – Porto-Novo,1984-02-28,1984,42
École Weber – Marseille,1984-03-15,1984,42
École Collet – Porto-Novo,1986-12-22,1986,39
École Poirier – Saint-Louis,1987-04-18,1987,39


### 7.2 · Les absences par mois en 2025-2026

`date_trunc('month', date_debut)` donne le 1er jour du mois de chaque absence. On regroupe sur ce mois en réutilisant le nom de la colonne calculée, `mois`, dans le `GROUP BY` (PostgreSQL l'accepte). `CAST(... AS date)` convertit le résultat (une date avec heure) en simple date, plus lisible. Le `BETWEEN` borne l'année scolaire (la table `absences` n'a pas de colonne `annee_scolaire_id`).

In [36]:
%%sql
SELECT CAST(date_trunc('month', date_debut) AS date) AS mois,
       COUNT(*) AS nb_absences
FROM absences
WHERE date_debut BETWEEN DATE '2025-09-01' AND DATE '2026-07-31'
GROUP BY mois
ORDER BY mois;

11 rows affected.

mois,nb_absences
2025-09-01,131
2025-10-01,121
2025-11-01,131
2025-12-01,132
2026-01-01,137
2026-02-01,88
2026-03-01,124
2026-04-01,104
2026-05-01,115
2026-06-01,100


Janvier 2026 est le mois le plus chargé (137 absences), février le plus calme (88).

### ✍️ Exercice 5 — Les enseignants les plus anciens

Pour préparer une remise de médailles, affichez le prénom, le nom, la spécialité, la date d'embauche et l'**ancienneté en années** des 5 enseignants embauchés depuis le plus longtemps.

In [37]:
%%sql
-- ✅ Solution
SELECT prenom, nom, specialite, date_embauche,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, date_embauche)) AS anciennete_ans
FROM enseignants
ORDER BY date_embauche
LIMIT 5;

5 rows affected.

prenom,nom,specialite,date_embauche,anciennete_ans
Raymond,Lévy,Mathématiques,1990-03-04,36
Robert,Rossi,Arts Plastiques,1990-10-01,36
Adrien,Blin,Mathématiques,1991-01-27,35
Claude,Delmas,Sciences de la Vie,1991-02-26,35
Yves,Deschamps,Sciences Économiques,1991-03-17,35


## 8. CASE WHEN : créer des catégories

`CASE WHEN condition THEN valeur ... ELSE valeur END` crée une colonne calculée selon des conditions testées **dans l'ordre** : la première condition vraie l'emporte. Transformons les moyennes du 1er trimestre 2025-2026 en **mentions** : 16 et plus « Très bien », 14 et plus « Bien », 12 et plus « Assez bien », 10 et plus « Passable », sinon « Insuffisant ».

### 8.1 · Une mention pour chaque bulletin

Grâce à l'ordre des tests, une moyenne de 15 n'est pas « Très bien » (15 < 16) mais s'arrête à « Bien » (15 ≥ 14) : inutile d'écrire `BETWEEN 14 AND 16`.

In [38]:
%%sql
SELECT eleve_id,
       moyenne_generale,
       CASE
           WHEN moyenne_generale >= 16 THEN 'Très bien'
           WHEN moyenne_generale >= 14 THEN 'Bien'
           WHEN moyenne_generale >= 12 THEN 'Assez bien'
           WHEN moyenne_generale >= 10 THEN 'Passable'
           ELSE 'Insuffisant'
       END AS mention
FROM bulletins
WHERE annee_scolaire_id = 7 AND trimestre = 1
ORDER BY eleve_id
LIMIT 10;

10 rows affected.

eleve_id,moyenne_generale,mention
201,6.24,Insuffisant
202,10.98,Passable
203,15.60,Bien
204,11.56,Passable
205,6.85,Insuffisant
206,10.71,Passable
207,13.57,Assez bien
208,11.95,Passable
209,12.12,Assez bien
211,13.60,Assez bien


### 8.2 · Combien d'élèves par mention ?

On regroupe sur la colonne calculée en réutilisant son alias `mention` dans le `GROUP BY`, comme pour `mois` en 7.2.

In [39]:
%%sql
SELECT CASE
           WHEN moyenne_generale >= 16 THEN 'Très bien'
           WHEN moyenne_generale >= 14 THEN 'Bien'
           WHEN moyenne_generale >= 12 THEN 'Assez bien'
           WHEN moyenne_generale >= 10 THEN 'Passable'
           ELSE 'Insuffisant'
       END AS mention,
       COUNT(*) AS nb_eleves
FROM bulletins
WHERE annee_scolaire_id = 7 AND trimestre = 1
GROUP BY mention
ORDER BY nb_eleves DESC;

5 rows affected.

mention,nb_eleves
Insuffisant,822
Passable,567
Assez bien,477
Bien,247
Très bien,96


> 💡 **Vérification** : la table `bulletins` contient déjà une colonne `appreciation`, remplie indépendamment de notre requête. Remplacez tout le `CASE ... END AS mention` par `appreciation` (et `GROUP BY mention` par `GROUP BY appreciation`) : vous obtenez exactement les mêmes effectifs (822, 567, 477, 247, 96). Recouper deux calculs indépendants est un excellent réflexe.

### ✍️ Exercice 6 — Petites, moyennes et grandes écoles

Classez les écoles selon leur capacité : `petite` (moins de 400 places), `moyenne` (de 400 à 799), `grande` (800 et plus). Combien d'écoles compte chaque catégorie ?

In [40]:
%%sql
-- ✅ Solution
SELECT CASE
           WHEN capacite_max < 400 THEN 'petite'
           WHEN capacite_max < 800 THEN 'moyenne'
           ELSE 'grande'
       END AS taille,
       COUNT(*) AS nb_ecoles
FROM ecoles
GROUP BY taille
ORDER BY nb_ecoles DESC;

3 rows affected.

taille,nb_ecoles
moyenne,12
grande,8
petite,4


## 9. Les vues : des requêtes enregistrées

Une **vue** est une requête enregistrée dans la base sous un nom. On l'interroge exactement comme une table, mais elle est **recalculée à chaque appel**. Notre base en propose deux :
- `v_classement_classe` : le rang de chaque élève dans sa classe, par trimestre (à partir des bulletins) ;
- `v_moyennes_eleves` : la moyenne de chaque élève dans chaque matière, calculée à partir du million de notes.

> ⚠️ Une vue peut cacher un calcul lourd : `v_moyennes_eleves` parcourt la table `notes`. Filtrez-la **toujours** (sur un élève ou une classe) ; n'écrivez jamais `SELECT * FROM v_moyennes_eleves` sans `WHERE`.

### 9.1 · Le podium de la classe la plus chargée

La classe n°597, repérée en partie 5 avec ses 63 élèves, est la 4ème A de l'École Collet – Porto-Novo. Qui sont ses 5 premiers au 1er trimestre ? La vue fait les jointures à notre place : on se contente de filtrer.

In [41]:
%%sql
SELECT rang, eleve_id, eleve_prenom, eleve_nom, moyenne_generale, appreciation
FROM v_classement_classe
WHERE classe_id = 597
  AND trimestre = 1
ORDER BY rang
LIMIT 5;

5 rows affected.

rang,eleve_id,eleve_prenom,eleve_nom,moyenne_generale,appreciation
1,1264,Margot,Bouvier,15.86,Bien
2,4663,Anne,Marchand,15.63,Bien
3,2970,André,Lucas,15.11,Bien
4,1338,Yves,Alves,14.78,Bien
5,4129,Christiane,Berger,14.14,Bien


### 9.2 · Les moyennes par matière d'une élève

L'élève classée 1re ci-dessus porte le numéro (`eleve_id`) 1264. Ses moyennes par matière sur toute l'année 2025-2026, avec la vue `v_moyennes_eleves` **filtrée** sur cette seule élève : PostgreSQL ne lit que ses notes, et la réponse arrive en moins d'une seconde au lieu d'un calcul sur plus d'un million de notes.

In [42]:
%%sql
SELECT matiere, coefficient_matiere, moyenne_matiere
FROM v_moyennes_eleves
WHERE eleve_id = 1264
  AND annee_scolaire = '2025-2026'
ORDER BY moyenne_matiere DESC;

7 rows affected.

matiere,coefficient_matiere,moyenne_matiere
Français,4.00,16.45
Anglais,2.00,15.80
Mathématiques,4.00,15.25
Éducation Physique,1.00,14.52
Sciences de la Vie,2.00,14.51
Physique-Chimie,3.00,14.41
Histoire-Géographie,2.00,14.40


## 10. Cas d'usage final : le tableau de bord 2025-2026

Tout est prêt pour répondre à la direction. Le livrable : **une ligne par pays** avec le nombre d'écoles, le nombre de classes ouvertes en 2025-2026, le nombre d'élèves inscrits en 2025-2026 et la moyenne générale du 1er trimestre. On construit la requête en deux étapes.

### 10.1 · Étape 1 : écoles et classes par pays

On descend la chaîne pays → villes → écoles → classes, en ne gardant que les classes de 2025-2026. Après les jointures, une même école apparaît sur plusieurs lignes (une par classe) : `COUNT(DISTINCT e.id)` compte chaque école **une seule fois**.

In [43]:
%%sql
SELECT p.nom                AS pays,
       COUNT(DISTINCT e.id) AS nb_ecoles,
       COUNT(DISTINCT c.id) AS nb_classes
FROM pays AS p
INNER JOIN villes  AS v ON v.pays_id = p.id
INNER JOIN ecoles  AS e ON e.ville_id = v.id
INNER JOIN classes AS c ON c.ecole_id = e.id
WHERE c.annee_scolaire_id = 7
GROUP BY p.nom
ORDER BY p.nom;

3 rows affected.

pays,nb_ecoles,nb_classes
Bénin,8,66
France,8,57
Sénégal,8,64


### 10.2 · Étape 2 : ajouter les élèves et la moyenne du 1er trimestre

On ajoute deux maillons avec `LEFT JOIN` :
- `inscriptions`, pour compter les élèves (`COUNT(DISTINCT i.eleve_id)`) ; le `LEFT JOIN` conserve les 56 classes vides de la partie 6 dans le nombre de classes ;
- `bulletins`, pour la moyenne : pour chaque élève inscrit, on prend **son** bulletin de 2025-2026 au 1er trimestre.

Les conditions sur les bulletins sont placées dans le `ON`, pas dans le `WHERE` : dans le `WHERE`, `b.trimestre = 1` éliminerait les lignes des classes vides (où `b.trimestre` vaut `NULL`) et fausserait le nombre de classes.

In [44]:
%%sql
SELECT p.nom                             AS pays,
       COUNT(DISTINCT e.id)              AS nb_ecoles,
       COUNT(DISTINCT c.id)              AS nb_classes,
       COUNT(DISTINCT i.eleve_id)        AS nb_eleves_inscrits,
       ROUND(AVG(b.moyenne_generale), 2) AS moyenne_t1
FROM pays AS p
INNER JOIN villes  AS v ON v.pays_id = p.id
INNER JOIN ecoles  AS e ON e.ville_id = v.id
INNER JOIN classes AS c ON c.ecole_id = e.id
LEFT JOIN inscriptions AS i ON i.classe_id = c.id
LEFT JOIN bulletins    AS b ON b.eleve_id = i.eleve_id
                           AND b.annee_scolaire_id = 7
                           AND b.trimestre = 1
WHERE c.annee_scolaire_id = 7
GROUP BY p.nom
ORDER BY nb_eleves_inscrits DESC;

3 rows affected.

pays,nb_ecoles,nb_classes,nb_eleves_inscrits,moyenne_t1
Bénin,8,66,827,11.00
France,8,57,726,10.83
Sénégal,8,64,656,10.94


**Lecture du tableau de bord** (chiffres obtenus lors de la préparation du TP) :
- 8 écoles dans chaque pays ; 187 classes ouvertes au total (66 au Bénin, 64 au Sénégal, 57 en France), dont 56 sans élève (partie 6) ;
- 2 209 élèves inscrits : 827 au Bénin, 726 en France, 656 au Sénégal ;
- des moyennes du 1er trimestre très proches, autour de 11/20 : de 10,83 en France à 11,00 au Bénin.

> 💡 **Recouper avant de livrer** : 827 + 726 + 656 = 2 209, exactement le nombre de bulletins du 1er trimestre trouvé en partie 5, et la moyenne globale de 10,92 se situe bien entre les moyennes des trois pays. Deux chemins différents, le même résultat : on peut livrer. Dans DataGrip, exportez ce résultat en CSV (fiche de connexion, étape 8) pour l'envoyer à la direction.

## 11. Bonus : WITH, RANK() et EXPLAIN

Trois outils pour aller plus loin, si vous avez terminé en avance.

### 11.1 · WITH : nommer une étape intermédiaire (CTE)

Une **CTE** (*Common Table Expression*) est une sous-requête à laquelle on donne un nom avec `WITH nom AS (...)`, puis qu'on utilise comme une table. La requête se lit alors de haut en bas, étape par étape. Question : quelles classes de 2025-2026 ont plus d'élèves inscrits que leur effectif maximal prévu (`effectif_max`) ?

In [45]:
%%sql
WITH inscrits_par_classe AS (
    SELECT classe_id, COUNT(*) AS nb_inscrits
    FROM inscriptions
    WHERE annee_scolaire_id = 7
    GROUP BY classe_id
)
SELECT e.nom AS ecole, c.nom AS classe, c.effectif_max, ipc.nb_inscrits
FROM inscrits_par_classe AS ipc
INNER JOIN classes AS c ON c.id = ipc.classe_id
INNER JOIN ecoles  AS e ON e.id = c.ecole_id
WHERE ipc.nb_inscrits > c.effectif_max
ORDER BY ipc.nb_inscrits DESC, c.id
LIMIT 10;

10 rows affected.

ecole,classe,effectif_max,nb_inscrits
École Collet – Porto-Novo,4ème A,33,63
École Perret – Cotonou,4ème A,43,61
École Morel – Abomey-Calavi,Tle A,40,61
École Rémy – Saint-Louis,3ème A,37,51
École Collet – Porto-Novo,3ème A,38,45
École Rémy – Saint-Louis,5ème A,28,43
École Poirier – Saint-Louis,1ère A,32,42
École Morin – Parakou,4ème A,39,41
École Samson – Ziguinchor,Tle A,38,41
École Denis – Paris,5ème A,37,39


Deuxième constat pour la direction : 12 classes sont en sureffectif (retirez le `LIMIT` pour les voir toutes). La 4ème A de l'École Collet – Porto-Novo accueille 63 élèves pour 33 places prévues.

### 11.2 · RANK() : une fonction de fenêtre

Une **fonction de fenêtre** calcule une valeur pour chaque ligne en regardant les autres lignes, sans les regrouper. `RANK() OVER (ORDER BY moyenne_t1 DESC)` donne le rang de chaque école : le rang 1 revient à la meilleure moyenne du 1er trimestre. Avec `PARTITION BY`, on obtiendrait un classement séparé dans chaque groupe (exemple dans « Pour aller plus loin »).

In [46]:
%%sql
WITH moyennes_ecoles AS (
    SELECT e.nom AS ecole,
           COUNT(*) AS nb_eleves,
           ROUND(AVG(b.moyenne_generale), 2) AS moyenne_t1
    FROM bulletins AS b
    INNER JOIN classes AS c ON c.id = b.classe_id
    INNER JOIN ecoles  AS e ON e.id = c.ecole_id
    WHERE b.annee_scolaire_id = 7 AND b.trimestre = 1
    GROUP BY e.nom
)
SELECT RANK() OVER (ORDER BY moyenne_t1 DESC) AS rang, ecole, nb_eleves, moyenne_t1
FROM moyennes_ecoles
ORDER BY rang, ecole
LIMIT 5;

5 rows affected.

rang,ecole,nb_eleves,moyenne_t1
1,École Gilles – Cotonou,4,13.02
2,École Muller – Parakou,2,12.52
3,École Guillou – Marseille,6,11.48
4,École Denis – Paris,135,11.47
5,École Laroche – Ziguinchor,87,11.24


> ⚠️ Regardez `nb_eleves` : les trois premières écoles n'ont que 2 à 6 élèves notés (les écoles primaires, presque vides). Une moyenne sur 4 élèves n'est pas comparable à une moyenne sur 150. Ajoutez `HAVING COUNT(*) >= 30` dans la CTE (après le `GROUP BY`) pour ne classer que les écoles suffisamment grandes.

### 11.3 · EXPLAIN : le plan d'exécution

`EXPLAIN` affiche le **plan d'exécution** choisi par PostgreSQL, **sans exécuter** la requête. Ici, on cherche les notes d'un élève : la colonne `eleve_id` possède un **index** (comme l'index à la fin d'un livre) et PostgreSQL saute directement aux bonnes lignes (`Index Scan` ou `Index Only Scan`).

In [47]:
%%sql
EXPLAIN
SELECT COUNT(*) FROM notes WHERE eleve_id = 1264;

3 rows affected.

QUERY PLAN
Aggregate (cost=5.85..5.86 rows=1 width=8)
-> Index Only Scan using idx_notes_eleve on notes (cost=0.43..5.31 rows=216 width=0)
Index Cond: (eleve_id = 1264)


### 11.4 · EXPLAIN sans index : lecture complète

Même question sur les notes égales à 20 : il n'y a pas d'index sur la colonne `note`, PostgreSQL doit lire toute la table (`Seq Scan`, lecture séquentielle, ici répartie sur plusieurs processus : `Parallel`).

In [48]:
%%sql
EXPLAIN
SELECT COUNT(*) FROM notes WHERE note = 20;

6 rows affected.

QUERY PLAN
Finalize Aggregate (cost=21060.08..21060.09 rows=1 width=8)
-> Gather (cost=21059.96..21060.07 rows=1 width=8)
Workers Planned: 1
-> Partial Aggregate (cost=20059.96..20059.97 rows=1 width=8)
-> Parallel Seq Scan on notes (cost=0.00..20000.13 rows=23933 width=0)
Filter: (note = '20'::numeric)


Comparez les valeurs `cost` (une estimation de l'effort, en unités arbitraires) des deux plans : quelques unités avec l'index, plusieurs milliers sans. C'est pourquoi on filtre de préférence sur des colonnes indexées, surtout sur une base partagée.

## Fin du TP : fermer la connexion

Que vous ayez fait le bonus ou non, terminez en libérant proprement les connexions ouvertes vers la base partagée avec `engine.dispose()`. C'est la deuxième et dernière cellule Python du notebook.

In [49]:
engine.dispose()  # ferme les connexions ouvertes vers la base partagée
print("Connexion fermée.")

Connexion fermée.


## Récapitulatif

**Ce que vous avez fait**
- exploré une base inconnue : `information_schema`, `COUNT(*)`, clés primaires et étrangères ;
- lu des données avec `SELECT`, `WHERE`, `ORDER BY`, `DISTINCT`, `LIMIT` ;
- résumé avec `COUNT`, `AVG`, `MIN`, `MAX`, `GROUP BY`, `HAVING` ;
- relié jusqu'à six tables avec `INNER JOIN` et `LEFT JOIN`, calculé sur des dates, créé des catégories avec `CASE WHEN`, interrogé des vues ;
- livré le tableau de bord 2025-2026 par pays, et repéré deux anomalies à signaler (56 classes sans élève, 12 classes en sureffectif).

**Les réflexes à retenir**
- compter (`COUNT(*)`) avant de récupérer, et toujours un `LIMIT` ou un filtre sur une base partagée ;
- ne sélectionner que les colonnes utiles : c'est plus rapide, et c'est la minimisation du RGPD ;
- `INNER JOIN` fait disparaître les lignes sans correspondance ; `LEFT JOIN` + `IS NULL` trouve les orphelins ;
- `WHERE` filtre les lignes, `HAVING` filtre les groupes ; `NULL` se teste avec `IS NULL` ;
- recouper un chiffre par deux chemins différents avant de le livrer ;
- ici, lecture seule : uniquement des `SELECT`.

## Pour aller plus loin

- Interroger la même base **depuis Python** (psycopg, pandas) pour enchaîner sur des traitements et des fichiers : c'est l'objet du notebook `02_postgres_python.ipynb`.
- Recalculer le rang des élèves dans chaque classe avec `RANK() OVER (PARTITION BY classe_id ORDER BY moyenne_generale DESC)` et le comparer à la colonne `rang` de la table `bulletins`.
- Dans DataGrip : afficher le diagramme du schéma `public`, puis essayer **Explain Plan** (clic droit sur une requête) pour voir le plan d'exécution sous forme graphique.
- Documentation PostgreSQL en français, chapitre « Le langage SQL » : https://docs.postgresql.fr/